# Practice Workshop — Week 11: Concatenating and merging

## Data

You will use monthly US data from FRED:

- `FRED_monthly_2000.csv` and `FRED_monthly_2010.csv`: the same columns, one decade each
- `UNRATE.csv`: unemployment rate (%), from 1948
- `FEDFUNDS.csv`: federal funds interest rate (%), from 1954

In [168]:
import pandas as pd

df_2000 = pd.read_csv('../../data/FRED/FRED_monthly_2000.csv')
df_2010 = pd.read_csv('../../data/FRED/FRED_monthly_2010.csv')
unrate = pd.read_csv('../../data/FRED/UNRATE.csv')
fedfunds = pd.read_csv('../../data/FRED/FEDFUNDS.csv')

## Exercise 1: Concatenate rows

1. How many rows should the result have if you stack `df_2000` and `df_2010`? Predict first.
2. Combine them into one DataFrame with `pd.concat()`.
3. Check for repeated index labels with `.index.duplicated().sum()`.
4. Make its row index run from 0 without repeated index labels.

Prediction: There is 120 rows in each of the dataframes, the two stacked should have 240

In [169]:
df = pd.concat([df_2000,df_2010])
df

,DATE,CPI,UNRATE,FEDFUNDS,REALRATE,LFPART
0,2000-01-01,169.3,4.0,5.4,2.7,67.3
1,2000-02-01,170.0,4.1,5.7,2.6,67.3
2,2000-03-01,171.0,4.0,5.8,2.7,67.3
3,2000-04-01,170.9,3.8,6.0,2.9,67.3
4,2000-05-01,171.2,4.0,6.3,2.8,67.1
...,...,...,...,...,...,...
115,2019-08-01,256.0,3.6,2.1,0.6,63.1
116,2019-09-01,256.4,3.5,2.0,0.3,63.2
117,2019-10-01,257.2,3.6,1.8,-0.0,63.3
118,2019-11-01,257.9,3.6,1.6,-0.2,63.3


In [170]:
df.index.duplicated().sum()

np.int64(120)

In [171]:
df = pd.concat([df_2000,df_2010],ignore_index=True)
df.duplicated().sum()

np.int64(0)

Now it runs from 0 to 240 without the duplicates

## Exercise 2: Merge tables

1. Check the first and last `DATE` in `unrate` and `fedfunds`. Do they cover the same months?
2. Merge `unrate` and `fedfunds` on `DATE` with an inner join. How many rows do you get?
3. Try an outer join. How many rows do you get now?
4. Explain the difference in row counts using what you found in step 1.

In [172]:
unrate.agg(['min','max'])

,DATE,UNRATE
min,1948-01-01,2.5
max,2026-06-01,14.8


In [173]:
fedfunds.agg(['min','max'])

,DATE,FEDFUNDS
min,1954-07-01,0.0
max,2026-07-01,19.1


Fedfunds is missing 5 years and 6 months compared to unrate

In [174]:
pd.merge(unrate,fedfunds)

,DATE,UNRATE,FEDFUNDS
0,1954-07-01,5.8,0.8
1,1954-08-01,6.0,1.2
2,1954-09-01,6.1,1.1
3,1954-10-01,5.7,0.8
4,1954-11-01,5.3,0.8
...,...,...,...
859,2026-02-01,4.4,3.6
860,2026-03-01,4.3,3.6
861,2026-04-01,4.3,3.6
862,2026-05-01,4.3,3.6


Inner join is standard and doesnt need a 'how=' argument. There are 864 rows. It keeps only the rows that both dataframes share on the column they are merged on. In this case they both have data from july 1954 and onwards so they are kept, but the unrate dataframe contained more rows that got left out

In [175]:
pd.merge(unrate,fedfunds,how='outer')

,DATE,UNRATE,FEDFUNDS
0,1948-01-01,3.4,NaN
1,1948-02-01,3.8,NaN
2,1948-03-01,4.0,NaN
3,1948-04-01,3.9,NaN
4,1948-05-01,3.5,NaN
...,...,...,...
938,2026-03-01,4.3,3.6
939,2026-04-01,4.3,3.6
940,2026-05-01,4.3,3.6
941,2026-06-01,4.2,3.6


Now there are 943 rows, thats because an outer join keeps all rows contained in any of the dataframes, in this case that means it keeps the rows that unrate has, but fedfunds does not. Thats why there are non-values in the fedfunds column

The data in the fedfunds starts later than unrate, thats why there are some months where unrate is recorded and not fedfunds

## Exercise 3: Handle missing values

In the outer-join result:

1. Find the rows with a missing `FEDFUNDS`. Which period do they cover?
2. Find the rows with a missing `UNRATE`. Why do you think these months are missing?
3. Fill missing `FEDFUNDS` with 0 in a copy of the result.
4. Does 0 mean the same thing as “unknown” here? Write one sentence explaining whether filling with 0 was a good choice.

**Think about:** Which join would you use if you only want months where both series are available?

In [176]:
outer = pd.merge(unrate,fedfunds,how='outer')

In [177]:
outer[outer['FEDFUNDS'].isna() == True].agg(['min','max'])

,DATE,UNRATE,FEDFUNDS
min,1948-01-01,2.5,NaN
max,1954-06-01,7.9,NaN


It covers the period between January 1948 and july 1954

In [178]:
outer[outer['UNRATE'].isna() == True].agg(['min','max'])

,DATE,UNRATE,FEDFUNDS
min,2025-10-01,NaN,3.6
max,2026-07-01,NaN,4.1


The unemployement rate is missing for september 2025 and june 2026, june 2026 is because that is the last month in the dataframe and the unemployement rate was not available at the time. The missing value for sept 2025 is harder to tell, could be a mistake in the data

In [179]:
outer = outer.fillna(0)

In [180]:
outer

,DATE,UNRATE,FEDFUNDS
0,1948-01-01,3.4,0.0
1,1948-02-01,3.8,0.0
2,1948-03-01,4.0,0.0
3,1948-04-01,3.9,0.0
4,1948-05-01,3.5,0.0
...,...,...,...
938,2026-03-01,4.3,3.6
939,2026-04-01,4.3,3.6
940,2026-05-01,4.3,3.6
941,2026-06-01,4.2,3.6


Filling the missing values is not the same as keeping it unknwon. This will affect the outcome when trying to compute average fedfunds, finding the min value etc. 

## Exercise 4 (challenge): Many-to-one merge

1. Make a table with the mean `SalePrice` for each `Neighborhood` (see Week 10).
2. Merge it back onto `houses` so each house gets its neighborhood’s mean price.
3. Explain why each neighborhood mean appears many times in the result.

In [181]:
houses = pd.read_csv('../../data/ames_houses.csv')

In [185]:
houses.head(1)

,LotArea,Neighborhood,OverallQuality,OverallCondition,YearBuilt,YearRemodeled,BuildingType,CentralAir,LivingArea,Bathrooms,Bedrooms,Fireplaces,SalePrice,YearSold,MonthSold,HasGarage
0,2951.2,NAmes,6,5,1960,1960,Single-family,Y,153.8,1,3,2,215.0,2010,5,1


In [187]:
nsp = houses.groupby('Neighborhood')['SalePrice'].mean().reset_index()
nsp = nsp.rename(columns={'SalePrice': 'MeanSalePrice'})

In [188]:
pd.merge(houses, nsp,how= 'outer',on='Neighborhood')

,LotArea,Neighborhood,OverallQuality,OverallCondition,YearBuilt,YearRemodeled,BuildingType,CentralAir,LivingArea,Bathrooms,Bedrooms,Fireplaces,SalePrice,YearSold,MonthSold,HasGarage,MeanSalePrice
0,297.5,Blmngtn,7,5,2006,2006,Townhouse,Y,106.4,2,2,0,160.0,2010,1,1,196.660714
1,295.6,Blmngtn,8,5,2004,2005,Townhouse,Y,117.9,2,2,1,192.0,2010,4,1,196.660714
2,342.2,Blmngtn,7,5,2007,2007,Townhouse,Y,144.5,2,2,1,174.0,2009,6,1,196.660714
3,337.7,Blmngtn,7,5,2007,2007,Townhouse,Y,145.8,2,1,1,175.9,2009,5,1,196.660714
4,295.6,Blmngtn,7,5,2005,2006,Townhouse,Y,144.5,2,2,1,192.5,2009,5,1,196.660714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,1651.5,Veenker,8,5,1981,1981,Single-family,Y,232.0,2,2,2,373.0,2007,3,1,248.316667
2926,1331.2,Veenker,7,4,1974,1974,Single-family,Y,245.8,2,3,4,260.0,2006,1,1,248.316667
2927,887.0,Veenker,8,5,1995,1996,Townhouse,Y,138.8,1,2,1,270.0,2006,4,1,248.316667
2928,1045.1,Veenker,6,6,1977,1977,Single-family,Y,112.2,1,3,1,165.0,2006,6,1,248.316667


Because in the 'houses' dataframe the neighborhoods appear several times and in the 'nsp' every neighborhood is assigned a mean price. The mean price is the same for the neighborhood every time it appears in the houses dataframe and since the two dataframes are merged on neighborhoods they will appear several times